# Работа с демонстрационной БД «Авиаперевозки» в PostgreSQL

Демонстрационная БД авиаперевозок развернута локально в Docker. Режим доступа — только на чтение.

In [1]:
import os
import json
import psycopg2
from psycopg2.extras import DictCursor

POSTGRESQL_HOST = os.environ.get('POSTGRESQL_HOST', 'localhost')
POSTGRESQL_USER = os.environ.get('POSTGRESQL_USER', 'postgres')
POSTGRESQL_PASSWORD = os.environ.get('POSTGRESQL_PASSWORD', 'postgres')


### Создание подключения к БД

In [2]:
conn = psycopg2.connect(
    dbname='demo',
    user=POSTGRESQL_USER,
    password=POSTGRESQL_PASSWORD,
    host=POSTGRESQL_HOST
)
cur = conn.cursor()
print("Подключение успешно:", conn.get_dsn_parameters())


Подключение успешно: {'user': 'postgres', 'channel_binding': 'prefer', 'dbname': 'demo', 'host': 'postgres', 'port': '5432', 'options': '', 'sslmode': 'prefer', 'sslcompression': '0', 'sslcertmode': 'allow', 'sslsni': '1', 'ssl_min_protocol_version': 'TLSv1.2', 'gssencmode': 'prefer', 'krbsrvname': 'postgres', 'gssdelegation': '0', 'target_session_attrs': 'any', 'load_balance_hosts': 'disable'}


### Пример: первые 5 записей из таблицы seats

In [3]:
query = 'SELECT * FROM seats LIMIT 5'
cur.execute(query)
records = cur.fetchall()
cur.close()
conn.close()
records


[('32N', '1A', 'Business'),
 ('32N', '1C', 'Business'),
 ('32N', '1D', 'Business'),
 ('32N', '1F', 'Business'),
 ('32N', '2A', 'Business')]

### Подключение через context manager

In [4]:
with psycopg2.connect(
    dbname='demo',
    user=POSTGRESQL_USER,
    password=POSTGRESQL_PASSWORD,
    host=POSTGRESQL_HOST
) as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT * FROM seats LIMIT 10')
        records = cur.fetchall()
records


[('32N', '1A', 'Business'),
 ('32N', '1C', 'Business'),
 ('32N', '1D', 'Business'),
 ('32N', '1F', 'Business'),
 ('32N', '2A', 'Business'),
 ('32N', '2C', 'Business'),
 ('32N', '2D', 'Business'),
 ('32N', '2F', 'Business'),
 ('32N', '3A', 'Business'),
 ('32N', '3C', 'Business')]

### Просмотр метаданных БД

In [5]:
queries = {
    '___DATABASES___': 'SELECT datname FROM pg_database',
    '___TABLES___': "SELECT relname FROM pg_class WHERE relkind='r' AND relname !~ '^(pg_|sql_)';",
    '\n___COLUMNS of seats___': "SELECT column_name FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = 'seats';"
}

with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    for name, query in queries.items():
        print('\n', name)
        with conn.cursor() as cur:
            cur.execute(query)
            for row in cur.fetchall():
                print(row)



 ___DATABASES___
('postgres',)
('demo',)
('template1',)
('template0',)
('test',)

 ___TABLES___
('airports_data',)
('seats',)
('airplanes_data',)
('tickets',)
('flights',)
('bookings',)
('routes',)
('segments',)
('boarding_passes',)

 
___COLUMNS of seats___
('airplane_code',)
('seat_no',)
('fare_conditions',)


In [6]:
# Загрузка jupysql для работы через %%sql
%load_ext sql

CONNECT_DATA = f'postgresql://{POSTGRESQL_USER}:{POSTGRESQL_PASSWORD}@{POSTGRESQL_HOST}/demo'


## Задание 1: Структура каждой таблицы

Для каждой таблицы выводим название колонок.

**Описание таблиц БД «Авиаперевозки»:**
- **airplanes_data** — справочник самолётов (код, модель, макс. дальность).
- **airports_data** — справочник аэропортов (код, название, город, координаты, часовой пояс).
- **boarding_passes** — посадочные талоны (номер билета, рейс, место, посадочный номер).
- **routes** — расписание маршрутов (`route_no`, аэропорты, самолёт, период `validity`).
- **bookings** — бронирования (номер, дата, итоговая сумма).
- **flights** — конкретные рейсы (`flight_id`, `route_no`, статус, расписание).
- **seats** — места в самолёте (код самолёта, номер места, класс обслуживания).
- **segments** — сегмент перелёта по билету (рейс, класс `fare_conditions`, цена `price`).
- **tickets** — билеты (номер, бронирование, пассажир).


In [7]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT relname FROM pg_class WHERE relkind='r' AND relname !~ '^(pg_|sql_)';")
        tables = [row[0] for row in cur.fetchall()]

    print(f"Таблицы в БД: {tables}\n")
    for table in tables:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT column_name, data_type FROM information_schema.columns WHERE table_name = %s ORDER BY ordinal_position;",
                (table,)
            )
            cols = cur.fetchall()
            print(f"\n📋 Таблица: {table}")
            print(f"  {'Колонка':<40} {'Тип'}")
            print("  " + "-"*60)
            for col_name, col_type in cols:
                print(f"  {col_name:<40} {col_type}")


Таблицы в БД: ['airports_data', 'seats', 'airplanes_data', 'tickets', 'flights', 'bookings', 'routes', 'segments', 'boarding_passes']


📋 Таблица: airports_data
  Колонка                                  Тип
  ------------------------------------------------------------
  airport_code                             character
  airport_name                             jsonb
  city                                     jsonb
  country                                  jsonb
  coordinates                              point
  timezone                                 text

📋 Таблица: seats
  Колонка                                  Тип
  ------------------------------------------------------------
  airplane_code                            character
  seat_no                                  text
  fare_conditions                          text

📋 Таблица: airplanes_data
  Колонка                                  Тип
  ------------------------------------------------------------
  airplane_code   

## Задание 2: Типы столбцов, количество записей, таблица с максимальным числом записей

In [8]:
table_counts = {}

with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT relname FROM pg_class WHERE relkind='r' AND relname !~ '^(pg_|sql_)';")
        tables = [row[0] for row in cur.fetchall()]

    for table in tables:
        with conn.cursor() as cur:
            # Типы столбцов
            cur.execute(
                "SELECT column_name, data_type FROM information_schema.columns WHERE table_name = %s ORDER BY ordinal_position;",
                (table,)
            )
            cols = cur.fetchall()

            # Количество записей
            cur.execute(f"SELECT COUNT(*) FROM {table}")
            count = cur.fetchone()[0]
            table_counts[table] = count

            print(f"\n📋 {table} — {count} записей")
            for col_name, col_type in cols:
                print(f"   {col_name}: {col_type}")

print("\n" + "="*50)
print("Словарь: кол-во записей по таблицам:")
print(json.dumps(table_counts, ensure_ascii=False, indent=2))

max_table = max(table_counts, key=table_counts.get)
print(f"\n🏆 Максимум записей: таблица '{max_table}' — {table_counts[max_table]:,} записей")



📋 airports_data — 5501 записей
   airport_code: character
   airport_name: jsonb
   city: jsonb
   country: jsonb
   coordinates: point
   timezone: text

📋 seats — 1741 записей
   airplane_code: character
   seat_no: text
   fare_conditions: text

📋 airplanes_data — 10 записей
   airplane_code: character
   model: jsonb
   range: integer
   speed: integer

📋 tickets — 2973937 записей
   ticket_no: text
   book_ref: character
   passenger_id: text
   passenger_name: text
   outbound: boolean

📋 flights — 21758 записей
   flight_id: integer
   route_no: text
   status: text
   scheduled_departure: timestamp with time zone
   scheduled_arrival: timestamp with time zone
   actual_departure: timestamp with time zone
   actual_arrival: timestamp with time zone

📋 bookings — 1292893 записей
   book_ref: character
   book_date: timestamp with time zone
   total_amount: numeric

📋 routes — 1162 записей
   route_no: text
   validity: tstzrange
   departure_airport: character
   arrival_airport

## Задание 3: Названия тарифов авиаперевозчиков

In [9]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT DISTINCT fare_conditions FROM segments ORDER BY fare_conditions;")
        fares = cur.fetchall()

print("Тарифы авиаперевозчиков:")
for fare in fares:
    print(f"  - {fare[0]}")


Тарифы авиаперевозчиков:
  - Business
  - Comfort
  - Economy


## Задание 4: Общая выручка по каждому тарифу

In [10]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                fare_conditions,
                SUM(price) AS total_revenue
            FROM segments
            GROUP BY fare_conditions
            ORDER BY total_revenue DESC;
        """)
        rows = cur.fetchall()

print(f"{'Тариф':<20} {'Выручка (руб.)':>20}")
print("-" * 42)
for fare, revenue in rows:
    print(f"{fare:<20} {revenue:>20,.2f}")


Тариф                      Выручка (руб.)
------------------------------------------
Economy                 17,696,017,750.00
Business                 4,939,418,500.00
Comfort                  1,938,495,325.00


## Задание 5: Тариф с максимальным доходом (SQL запрос)

In [11]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT fare_conditions, SUM(price) AS total_revenue
            FROM segments
            GROUP BY fare_conditions
            ORDER BY total_revenue DESC
            LIMIT 1;
        """)
        row = cur.fetchone()

print(f"Тариф с максимальным доходом: {row[0]}")
print(f"Общая выручка: {row[1]:,.2f} руб.")


Тариф с максимальным доходом: Economy
Общая выручка: 17,696,017,750.00 руб.


# Время выполнения запросов

Разные запросы требуют разное время на выполнение.

## Задание 5 (b): Модель самолёта с минимальной максимальной дальностью полёта — 2 способа

In [12]:
import time

with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:

    # --- Способ 1: подзапрос ---
    start = time.time()
    with conn.cursor() as cur:
        cur.execute("""
            SELECT model, range
            FROM airplanes_data
            WHERE range = (SELECT MIN(range) FROM airplanes_data);
        """)
        result1 = cur.fetchall()
    t1 = time.time() - start

    # --- Способ 2: ORDER BY + LIMIT ---
    start = time.time()
    with conn.cursor() as cur:
        cur.execute("""
            SELECT model, range
            FROM airplanes_data
            ORDER BY range ASC
            LIMIT 1;
        """)
        result2 = cur.fetchall()
    t2 = time.time() - start

print("Способ 1 (подзапрос MIN):")
for row in result1:
    print(f"  Модель: {row[0]}, Дальность: {row[1]} км")
print(f"  Время: {t1*1000:.3f} мс\n")

print("Способ 2 (ORDER BY + LIMIT):")
for row in result2:
    print(f"  Модель: {row[0]}, Дальность: {row[1]} км")
print(f"  Время: {t2*1000:.3f} мс\n")

print("Вывод: ORDER BY + LIMIT обычно быстрее на небольших таблицах,")
print("т.к. может использовать индекс и остановиться на первой записи.")
print("Подзапрос с MIN сканирует таблицу дважды (агрегация + фильтр).")


Способ 1 (подзапрос MIN):
  Модель: {'en': 'Bombardier CRJ700', 'ru': 'Бомбардье CRJ700'}, Дальность: 3100 км
  Время: 1.992 мс

Способ 2 (ORDER BY + LIMIT):
  Модель: {'en': 'Bombardier CRJ700', 'ru': 'Бомбардье CRJ700'}, Дальность: 3100 км
  Время: 0.546 мс

Вывод: ORDER BY + LIMIT обычно быстрее на небольших таблицах,
т.к. может использовать индекс и остановиться на первой записи.
Подзапрос с MIN сканирует таблицу дважды (агрегация + фильтр).


## Задание 6: Количество рейсов с максимальной длительностью полёта

In [13]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                MAX(scheduled_arrival - scheduled_departure) AS max_duration,
                COUNT(*) FILTER (
                    WHERE (scheduled_arrival - scheduled_departure) =
                          (SELECT MAX(scheduled_arrival - scheduled_departure) FROM flights)
                ) AS flights_count
            FROM flights;
        """)
        row = cur.fetchone()

print(f"Максимальная длительность полёта: {row[0]}")
print(f"Количество рейсов с максимальной длительностью: {row[1]}")


Максимальная длительность полёта: 19:55:00
Количество рейсов с максимальной длительностью: 9


## Задание 7: Уникальные маршруты с максимальной длительностью полёта

In [14]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            WITH max_duration_routes AS (
                SELECT
                    r.departure_airport,
                    r.arrival_airport,
                    MAX(f.scheduled_arrival - f.scheduled_departure) AS max_route_duration
                FROM flights f
                JOIN routes r ON r.route_no = f.route_no AND f.scheduled_departure <@ r.validity
                GROUP BY r.departure_airport, r.arrival_airport
            ),
            global_max AS (
                SELECT MAX(max_route_duration) AS global_max_dur FROM max_duration_routes
            )
            SELECT
                mdr.max_route_duration                       AS duration,
                dep.airport_name->>'ru'                      AS departure_airport_name,
                dep.city->>'ru'                              AS departure_city,
                arr.airport_name->>'ru'                      AS arrival_airport_name,
                arr.city->>'ru'                              AS arrival_city
            FROM max_duration_routes mdr
            JOIN global_max gm ON mdr.max_route_duration = gm.global_max_dur
            JOIN airports_data dep ON mdr.departure_airport = dep.airport_code
            JOIN airports_data arr ON mdr.arrival_airport   = arr.airport_code
            ORDER BY departure_airport_name;
        """)
        rows = cur.fetchall()

print(f"{'Длит.':<12} {'Аэропорт отправления':<35} {'Город отпр.':<20} {'Аэропорт прибытия':<35} {'Город приб.'}")
print("-"*130)
for row in rows:
    print(f"{str(row[0]):<12} {row[1]:<35} {row[2]:<20} {row[3]:<35} {row[4]}")


Длит.        Аэропорт отправления                Город отпр.          Аэропорт прибытия                   Город приб.
----------------------------------------------------------------------------------------------------------------------------------
19:55:00     Артуро Мерино Бенитес               Сантьяго             Чхатрапати Шиваджи                  Мумбаи
19:55:00     Чхатрапати Шиваджи                  Мумбаи               Артуро Мерино Бенитес               Сантьяго


## Задание 8: Аэропорт с максимальной нагрузкой (отправления + прибытия)

In [15]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            WITH airport_load AS (
                SELECT r.departure_airport AS airport_code, COUNT(*) AS cnt
                FROM flights f
                JOIN routes r ON r.route_no = f.route_no AND f.scheduled_departure <@ r.validity
                GROUP BY r.departure_airport
                UNION ALL
                SELECT r.arrival_airport AS airport_code, COUNT(*) AS cnt
                FROM flights f
                JOIN routes r ON r.route_no = f.route_no AND f.scheduled_departure <@ r.validity
                GROUP BY r.arrival_airport
            ),
            total_load AS (
                SELECT airport_code, SUM(cnt) AS total FROM airport_load GROUP BY airport_code
            )
            SELECT
                tl.airport_code,
                ad.airport_name->>'ru' AS airport_name,
                ad.city->>'ru'         AS city,
                tl.total               AS total_flights
            FROM total_load tl
            JOIN airports_data ad ON tl.airport_code = ad.airport_code
            ORDER BY tl.total DESC
            LIMIT 1;
        """)
        row = cur.fetchone()

print(f"Аэропорт с максимальной нагрузкой:")
print(f"  Код:              {row[0]}")
print(f"  Название:         {row[1]}")
print(f"  Город:            {row[2]}")
print(f"  Всего рейсов:     {row[3]:,}")


Аэропорт с максимальной нагрузкой:
  Код:              FRA
  Название:         Франкфурт-на-Майне
  Город:            Франкфурт-на-Майне
  Всего рейсов:     1,429


## Задание 9: Среднее количество мест в самолётах по классу обслуживания

In [16]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                fare_conditions,
                ROUND(AVG(seat_count), 2) AS avg_seat_count
            FROM (
                SELECT airplane_code, fare_conditions, COUNT(*) AS seat_count
                FROM seats
                GROUP BY airplane_code, fare_conditions
            ) sub
            GROUP BY fare_conditions
            ORDER BY fare_conditions;
        """)
        rows = cur.fetchall()

print(f"{'fare_conditions':<20} {'avg_seat_count':>15}")
print("-"*37)
for fare, avg in rows:
    print(f"{fare:<20} {avg:>15}")


fare_conditions       avg_seat_count
-------------------------------------
Business                       25.88
Comfort                        27.25
Economy                       178.13


## Задание 10: Самый дорогой перелёт + EXPLAIN ANALYZE

In [17]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            WITH flight_revenue AS (
                SELECT tf.flight_id, SUM(tf.price) AS final_amount
                FROM segments tf
                GROUP BY tf.flight_id
            ),
            max_rev AS (
                SELECT MAX(final_amount) AS max_amount FROM flight_revenue
            )
            SELECT
                fr.flight_id,
                fr.final_amount,
                dep.airport_name->>'ru' AS departure_airport,
                dep.city->>'ru'         AS departure_city,
                arr.airport_name->>'ru' AS arrival_airport,
                arr.city->>'ru'         AS arrival_city
            FROM flight_revenue fr
            JOIN max_rev      mr  ON fr.final_amount = mr.max_amount
            JOIN flights       f   ON f.flight_id     = fr.flight_id
            JOIN routes          r   ON r.route_no = f.route_no AND f.scheduled_departure <@ r.validity
            JOIN airports_data dep ON r.departure_airport = dep.airport_code
            JOIN airports_data arr ON r.arrival_airport   = arr.airport_code
            ORDER BY fr.flight_id;
        """)
        rows = cur.fetchall()

print("Самые дорогие перелёты:")
print(f"{'flight_id':<12} {'final_amount':>15} {'departure_airport':<35} {'departure_city':<20} {'arrival_airport':<35} {'arrival_city'}")
print("-"*130)
for row in rows:
    print(f"{row[0]:<12} {float(row[1]):>15,.2f} {row[2]:<35} {row[3]:<20} {row[4]:<35} {row[5]}")
print(f"\nВсего рейсов с максимальной суммой выручки: {len(rows)}")


Самые дорогие перелёты:
flight_id       final_amount departure_airport                   departure_city       arrival_airport                     arrival_city
----------------------------------------------------------------------------------------------------------------------------------
16679          22,532,100.00 Окленд                              Окленд               Шарлотт Дуглас                      Шарлотт

Всего рейсов с максимальной суммой выручки: 1


In [18]:
# EXPLAIN ANALYZE для оценки плана выполнения запроса
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            EXPLAIN ANALYZE
            WITH flight_revenue AS (
                SELECT tf.flight_id, SUM(tf.price) AS final_amount
                FROM segments tf
                GROUP BY tf.flight_id
            ),
            max_rev AS (
                SELECT MAX(final_amount) AS max_amount FROM flight_revenue
            )
            SELECT
                fr.flight_id,
                fr.final_amount,
                dep.airport_name->>'ru' AS departure_airport,
                dep.city->>'ru'         AS departure_city,
                arr.airport_name->>'ru' AS arrival_airport,
                arr.city->>'ru'         AS arrival_city
            FROM flight_revenue fr
            JOIN max_rev      mr  ON fr.final_amount = mr.max_amount
            JOIN flights       f   ON f.flight_id     = fr.flight_id
            JOIN routes          r   ON r.route_no = f.route_no AND f.scheduled_departure <@ r.validity
            JOIN airports_data dep ON r.departure_airport = dep.airport_code
            JOIN airports_data arr ON r.arrival_airport   = arr.airport_code
            ORDER BY fr.flight_id;
        """)
        plan = cur.fetchall()

print("EXPLAIN ANALYZE:")
for line in plan:
    print(line[0])


EXPLAIN ANALYZE:
Sort  (cost=63137.29..63137.29 rows=1 width=164) (actual time=706.953..707.063 rows=1 loops=1)
  Sort Key: fr.flight_id
  Sort Method: quicksort  Memory: 25kB
  CTE flight_revenue
    ->  Finalize HashAggregate  (cost=62169.95..62369.59 rows=15971 width=36) (actual time=680.319..692.390 rows=20793 loops=1)
          Group Key: tf.flight_id
          Batches: 5  Memory Usage: 9265kB  Disk Usage: 264kB
          ->  Gather  (cost=58536.55..61930.38 rows=31942 width=36) (actual time=600.526..627.914 rows=61121 loops=1)
                Workers Planned: 2
                Workers Launched: 2
                ->  Partial HashAggregate  (cost=57536.55..57736.18 rows=15971 width=36) (actual time=596.068..611.319 rows=20374 loops=3)
                      Group Key: tf.flight_id
                      Batches: 5  Memory Usage: 8241kB  Disk Usage: 240kB
                      Worker 0:  Batches: 5  Memory Usage: 8241kB  Disk Usage: 248kB
                      Worker 1:  Batches: 5  M

In [19]:
# Применяем индексы для оптимизации (выполнять с правами superuser)
# Индексы на часто используемые в JOIN и WHERE столбцы:
optimizations = """
-- Рекомендации по оптимизации запроса:
-- 1. Если видим Seq Scan на segments -> создаём индекс:
--    В демо 2025 уже есть индекс segments_flight_id_idx на flight_id.
--    При необходимости: индекс на segments(price).

-- 2. Если видим Seq Scan на flights -> создаём индекс:
--    CREATE INDEX IF NOT EXISTS idx_flights_id ON flights(flight_id);

-- 3. Hash Join эффективнее Nested Loop для больших таблиц:
--    SET enable_nestloop = off;  -- временно для тестирования
"""
print(optimizations)



-- Рекомендации по оптимизации запроса:
-- 1. Если видим Seq Scan на segments -> создаём индекс:
--    В демо 2025 уже есть индекс segments_flight_id_idx на flight_id.
--    При необходимости: индекс на segments(price).

-- 2. Если видим Seq Scan на flights -> создаём индекс:
--    CREATE INDEX IF NOT EXISTS idx_flights_id ON flights(flight_id);

-- 3. Hash Join эффективнее Nested Loop для больших таблиц:
--    SET enable_nestloop = off;  -- временно для тестирования



## Дополнительное задание: 3 запроса для выявления узких мест авиаперевозчика

In [20]:
with psycopg2.connect(dbname='demo', user=POSTGRESQL_USER, password=POSTGRESQL_PASSWORD, host=POSTGRESQL_HOST) as conn:

    # --- Запрос 1: Рейсы с большим процентом незаполненных мест ---
    print("="*60)
    print("1. Рейсы с наибольшим процентом незаполненных мест")
    print("   (низкая загрузка — потеря выручки)")
    print("="*60)
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                f.route_no,
                r.departure_airport,
                r.arrival_airport,
                a.model->>'ru'                         AS aircraft_model,
                COUNT(s.seat_no)                       AS total_seats,
                COUNT(bp.seat_no)                      AS occupied_seats,
                ROUND(
                    COUNT(bp.seat_no)::numeric / COUNT(s.seat_no) * 100, 1
                )                                      AS occupancy_pct
            FROM flights f
            JOIN routes r ON r.route_no = f.route_no AND f.scheduled_departure <@ r.validity
            JOIN airplanes_data a  ON r.airplane_code = a.airplane_code
            JOIN seats s           ON s.airplane_code  = r.airplane_code
            LEFT JOIN boarding_passes bp
                ON bp.flight_id = f.flight_id AND bp.seat_no = s.seat_no
            WHERE f.status = 'Arrived'
            GROUP BY f.flight_id, f.route_no, r.departure_airport,
                     r.arrival_airport, a.model
            ORDER BY occupancy_pct ASC
            LIMIT 10;
        """)
        rows = cur.fetchall()
    print(f"{'Рейс':<10} {'Откуда':<8} {'Куда':<8} {'Самолёт':<25} {'Мест':>6} {'Заняты':>7} {'Загр.%':>8}")
    print("-"*80)
    for row in rows:
        print(f"{row[0]:<10} {row[1]:<8} {row[2]:<8} {row[3]:<25} {row[4]:>6} {row[5]:>7} {float(row[6]):>7.1f}%")

    # --- Запрос 2: Аэропорты с наибольшим количеством задержанных/отменённых рейсов ---
    print("\n" + "="*60)
    print("2. Аэропорты с наибольшим % отменённых/задержанных рейсов")
    print("   (проблемы с расписанием / инфраструктурой)")
    print("="*60)
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                ad.airport_name->>'ru' AS airport,
                ad.city->>'ru'         AS city,
                COUNT(*)               AS total_departures,
                COUNT(*) FILTER (WHERE f.status IN ('Cancelled', 'Delayed')) AS problem_flights,
                ROUND(
                    COUNT(*) FILTER (WHERE f.status IN ('Cancelled', 'Delayed'))::numeric
                    / COUNT(*) * 100, 1
                ) AS problem_pct
            FROM flights f
            JOIN routes r ON r.route_no = f.route_no AND f.scheduled_departure <@ r.validity
            JOIN airports_data ad ON r.departure_airport = ad.airport_code
            GROUP BY ad.airport_code, ad.airport_name, ad.city
            HAVING COUNT(*) > 10
            ORDER BY problem_pct DESC
            LIMIT 10;
        """)
        rows = cur.fetchall()
    print(f"{'Аэропорт':<35} {'Город':<20} {'Рейсов':>8} {'Проблем':>8} {'%':>8}")
    print("-"*85)
    for row in rows:
        print(f"{row[0]:<35} {row[1]:<20} {row[2]:>8} {row[3]:>8} {float(row[4]):>7.1f}%")

    # --- Запрос 3: Маршруты с наибольшим средним отклонением от расписания ---
    print("\n" + "="*60)
    print("3. Маршруты с наибольшим средним опозданием")
    print("   (систематические задержки — узкое место)")
    print("="*60)
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                r.departure_airport,
                r.arrival_airport,
                dep.city->>'ru'  AS dep_city,
                arr.city->>'ru'  AS arr_city,
                COUNT(*)         AS flight_count,
                AVG(EXTRACT(EPOCH FROM (f.actual_departure - f.scheduled_departure))/60)::int
                                 AS avg_dep_delay_min
            FROM flights f
            JOIN routes r ON r.route_no = f.route_no AND f.scheduled_departure <@ r.validity
            JOIN airports_data dep ON r.departure_airport = dep.airport_code
            JOIN airports_data arr ON r.arrival_airport   = arr.airport_code
            WHERE f.actual_departure IS NOT NULL
              AND f.actual_departure > f.scheduled_departure
            GROUP BY r.departure_airport, r.arrival_airport, dep.city, arr.city
            HAVING COUNT(*) >= 5
            ORDER BY avg_dep_delay_min DESC
            LIMIT 10;
        """)
        rows = cur.fetchall()
    print(f"{'Откуда':<8} {'Куда':<8} {'Из города':<20} {'В город':<20} {'Рейсов':>7} {'Ср.задерж.(мин)':>17}")
    print("-"*87)
    for row in rows:
        print(f"{row[0]:<8} {row[1]:<8} {row[2]:<20} {row[3]:<20} {row[4]:>7} {row[5]:>17}")


1. Рейсы с наибольшим процентом незаполненных мест
   (низкая загрузка — потеря выручки)
Рейс       Откуда   Куда     Самолёт                     Мест  Заняты   Загр.%
--------------------------------------------------------------------------------
PG0351     MXP      LHR      Аэробус A320neo              166       2     1.2%
PG0131     SCL      AKL      Боинг 787-9 Dreamliner       257       3     1.2%
PG0041     KTM      KMG      Эмбраэр E170                  78       1     1.3%
PG0365     ORY      LHR      Боинг 777-300ER              404       7     1.7%
PG0171     CTU      WUH      Боинг 777-300ER              404       8     2.0%
PG0132     AKL      SCL      Боинг 787-9 Dreamliner       257       6     2.3%
PG0351     MXP      LHR      Аэробус A320neo              166       4     2.4%
PG0365     ORY      LHR      Боинг 777-300ER              404      10     2.5%
PG0130     BOM      SCL      Аэробус A350-1000            325       9     2.8%
PG0092     PKX      AKL      Аэробус A33

In [21]:
# Задержка вылета по каждому рейсу (интервал времени)
with psycopg2.connect(
    dbname='demo',
    user=POSTGRESQL_USER,
    password=POSTGRESQL_PASSWORD,
    host=POSTGRESQL_HOST,
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                flight_id,
                actual_departure - scheduled_departure AS delay_interval
            FROM flights
            WHERE actual_departure IS NOT NULL
              AND scheduled_departure IS NOT NULL;
        """)
        rows = cur.fetchall()

print(f"Найдено рейсов: {len(rows)}")
for row in rows[:20]:
    print(row)

Найдено рейсов: 10986
(72, datetime.timedelta(seconds=268, microseconds=291593))
(2, datetime.timedelta(seconds=289, microseconds=838375))
(35, datetime.timedelta(seconds=426, microseconds=667592))
(89, datetime.timedelta(seconds=488, microseconds=884115))
(112, datetime.timedelta(seconds=133, microseconds=217186))
(64, datetime.timedelta(seconds=273, microseconds=288235))
(23, datetime.timedelta(seconds=244, microseconds=453425))
(116, datetime.timedelta(seconds=204, microseconds=5559))
(29, datetime.timedelta(seconds=352, microseconds=152011))
(34, datetime.timedelta(seconds=253, microseconds=285508))
(21, datetime.timedelta(seconds=461, microseconds=521981))
(92, datetime.timedelta(seconds=408, microseconds=704669))
(25, datetime.timedelta(seconds=437, microseconds=608250))
(54, datetime.timedelta(seconds=282, microseconds=324930))
(26, datetime.timedelta(seconds=393, microseconds=430120))
(58, datetime.timedelta(seconds=547, microseconds=346904))
(76, datetime.timedelta(seconds=94,